# Bauen wir einen Spamklassifikator

Ein Spamfilter ist eine klassische Aufgabe der Textklassifikation: Anhand des Texts einer Nachricht wird entschieden, ob sie Spam oder eine echte Nachricht (Ham) ist. In diesem Notebook bauen wir Schritt für Schritt einen Spamfilter. Wir wandeln Rohtext in numerische Features um, trainieren einen Klassifikator mit logistischer Regression und verbessern die Features dann mit TF-IDF, Stemming und Lemmatisierung. Dabei vergleichen wir jeweils, wie sich das auf den Score auswirkt.

## Lernziele

Am Ende dieses Notebooks kannst du:

- mit `CountVectorizer` und `TfidfVectorizer` numerische Features aus Rohtext erstellen.
- einen Spamklassifikator mit logistischer Regression trainieren und mit dem AUC-Score evaluieren.
- Stemming und Lemmatisierung als Vorverarbeitungsschritte anwenden und ihre Wirkung vergleichen.
- ein lineares Modell anhand der Wörter mit den größten und kleinsten Koeffizienten interpretieren.

Wir verwenden Daten aus `SMS Spam Collection v. 1`, die so beschrieben werden:

> eine öffentliche Zusammenstellung von SMS-Nachrichten mit Labels, die für die Forschung zu Spam auf Mobiltelefonen gesammelt wurden. Sie enthält eine Sammlung aus 5574 englischen, echten und nicht kodierten Nachrichten, die als legitim (Ham) oder Spam gekennzeichnet sind.

([Quelle](http://www.dt.fee.unicamp.br/~tiago/smsspamcollection/))

## Bibliotheken und Daten einrichten

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

import nltk

nltk.download("wordnet")
nltk.download("omw-1.4")

In [ ]:
# Daten laden
data = pd.read_csv(
    "data/SMSSpamCollection.txt",
    encoding="utf-8",
    header=None,
    delimiter="\t",
    names=["target", "text"],
)

# Zielvariable kodieren
data["target"] = np.where(data["target"] == "spam", 1, 0)

In [ ]:
# Eine Stichprobe unserer Daten ansehen
data.sample(3)

## Ein kurzer Blick auf die Daten

In [ ]:
print(
    "Dataset contains {} instances of {} variables.".format(
        data.shape[0], data.shape[1]
    )
)

print(
    "It contains {} spam messages ({:.1%} of all).".format(
        data[data.target == 1].shape[0],
        data[data.target == 1].shape[0] / data.shape[0],
    )
)

In [ ]:
## Zufällige Stichproben von Texten aus beiden Klassen ausgeben, d. h. Spam und Nicht-Spam
print(
    "Examples of spam SMS: \n    {}\n    {}".format(
        data[data.target == 1].sample(1).text.iloc[0],
        data[data.target == 1].sample(1).text.iloc[0],
    )
)
print(
    "\nExamples of non-spam SMS: \n    {}\n    {}".format(
        data[data.target == 0].sample(1).text.iloc[0],
        data[data.target == 0].sample(1).text.iloc[0],
    )
)

## Spamklassifikation

Hier bauen wir einen „Vanilla“-Klassifikator, ohne uns allzu viele Gedanken darüber zu machen, wie die eigentlichen Nachrichten, ob Spam oder nicht, aussehen. Um dein Modell zu verbessern, kannst du dir die Daten natürlich genauer ansehen und sie detaillierter untersuchen.

In [ ]:
# Datensatz in Trainings- und Testdatensatz aufteilen
X_train, X_test, y_train, y_test = train_test_split(
    data["text"], data["target"], random_state=0
)

In [ ]:
X_train.head()

### CountVectorizer

Wie du gesehen hast, ist unsere X-Variable einfach nur Text == ein String. Damit kann kein Klassifikator umgehen, also müssen wir den Text für das Modell zugänglich machen. Dazu können wir den Text so umwandeln, dass jedes Wort ein eigenes Feature ist, und zählen, wie oft dieses Wort in der SMS vorkommt. Das geht mit dem [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) von scikit-learn. Er wandelt unseren Text um und gibt eine dünnbesetzte Matrix zurück. Das Vokabular eines englischen Texts ist ziemlich umfangreich, während du in einer SMS nur eine kleine Teilmenge der Wörter verwendest. Deshalb spart es Arbeitsspeicher, diese Feature-Matrix als dünnbesetzte Matrix zu speichern.



In [ ]:
# CountVectorizer an die Trainingsdaten anpassen
vect = CountVectorizer().fit(X_train)

# Dokumente der Trainingsdaten in eine Dokument-Term-Matrix umwandeln
X_train_vectorized = vect.transform(X_train)
# print("X_train_vectorized: ", X_train_vectorized)

In [ ]:
print("X_train shape = {}".format(X_train.shape))
print("Vocabulary length = {}".format(len(vect.vocabulary_)))

In 4179 Nachrichten haben wir also 7546 verschiedene Wörter gefunden.

In [ ]:
# Sehen wir uns die Liste unseres Vokabulars an (alphabetisch sortiert)
# Sieht sie so aus, wie du erwartet hast?
sorted(vect.vocabulary_.items(), key=lambda x: x[1])[:20]

In [ ]:
# Wir können auch die neu erstellte Feature-Matrix ausgeben
# Hinweis: Du siehst, dass es eine dünnbesetzte Matrix mit vielen 0-Werten ist.
# Mit .toarray() wird die komprimierte Form der dünnbesetzten Matrix in ein normales NumPy-Array umgewandelt
print(X_train_vectorized.toarray())

Mit den vektorisierten Daten können wir unser erstes Modell trainieren.

In [ ]:
# Modell trainieren
model = LogisticRegression(max_iter=1500)
model.fit(X_train_vectorized, y_train)

# Vorhersage für die transformierten Testdokumente
predictions = model.predict(vect.transform(X_test))
predict_probab = model.predict_proba(vect.transform(X_test))[:, 1]

print("AUC = {:.3f}".format(roc_auc_score(y_test, predict_probab)))

**Ergebnis:** Das zählbasierte Modell erreicht auf dem zurückgehaltenen Testdatensatz eine AUC von etwa 0,985. Die AUC (area under the ROC curve) misst, wie gut das Modell Spam über alle Entscheidungsschwellenwerte hinweg höher einstuft als Ham, wobei 1,0 perfekt ist und 0,5 zufälligem Raten entspricht. Ein Wert nahe 0,985 bedeutet, dass die einfachen Wortanzahlen die beiden Klassen bereits gut trennen.

Der Datensatz ist unausgewogen (nur etwa 13% der Nachrichten sind Spam), daher würde die reine Accuracy selbst bei einem schlechten Modell hoch ausfallen. Die AUC ist hier ein besserer Anhaltspunkt, weil sie kein Modell belohnt, das einfach die Mehrheitsklasse vorhersagt.

Sehen wir uns an, welche Wörter die höchsten und niedrigsten Koeffizienten haben.

Erinnere dich an die Sigmoidfunktion (logistische Funktion).
Welcher Klasse werden Beobachtungen zugeordnet, wenn sie Wörter mit hohen Koeffizienten enthalten? Und welcher Klasse werden sie zugeordnet, wenn sie Wörter mit stark negativen Koeffizienten enthalten?

In [ ]:
# Feature-Namen als NumPy-Array abrufen
feature_names = np.array(vect.get_feature_names_out())

# Koeffizienten des Modells sortieren (vom niedrigsten zum höchsten Wert)
sorted_coef_index = model.coef_[0].argsort()

# Die 10 kleinsten und die 10 größten Koeffizienten finden
# Die 10 größten Koeffizienten werden mit [:-11:-1] indexiert,
# sodass die Liste vom größten zum kleinsten Wert sortiert ist
print("Smallest Coefs:\n{}\n".format(feature_names[sorted_coef_index[:10]]))
print("Largest Coefs: \n{}".format(feature_names[sorted_coef_index[:-11:-1]]))

Das erste Modell erreicht auf dem Testdatensatz bereits eine AUC von etwa 0,985, ein starker Score für eine einfache Repräsentation auf Basis von Wortanzahlen. Als Nächstes testen wir, ob TF-IDF, eine andere Art, Text in Zahlen umzuwandeln, besser abschneidet.



### TF-IDF

TF-IDF steht für **Term Frequency - Inverse Document Frequency**.

TF-IDF misst, wie wichtig ein Wort für ein Dokument in einer Sammlung von Texten ist (in unserem Fall alle SMS, die wir gesammelt haben). Ein Wort, das in einem Dokument häufig vorkommt und auch im Korpus häufig ist, ist für das Dokument weniger wichtig als ein Wort, das in einem Dokument häufig vorkommt, im Korpus aber nicht.




In [ ]:
# TfidfVectorizer an die Trainingsdaten anpassen, mit einer minimalen Dokumenthäufigkeit von 15
# Das heißt, ein Wort muss in mindestens 15 SMS verwendet worden sein
vect = TfidfVectorizer(min_df=15).fit(X_train)

# Dokumente der Trainingsdaten in eine Dokument-Term-Matrix umwandeln
X_train_vectorized = vect.transform(X_train)

# Sehen wir uns einige der Wörter an, die mit dieser Methode gesammelt wurden
sorted(vect.vocabulary_.items(), key=lambda x: x[1])[10:30]

In [ ]:
# Wie viele Wörter in mehr als 15 Textnachrichten vorkommen
len(sorted(vect.vocabulary_.items(), key=lambda x: x[1]))

Wir können prüfen, welche Wörter die größten TF-IDF-Werte für die Texte erzeugt haben.

In [ ]:
# Alle Feature-Namen == Wörter in einem Array speichern
feature_names = np.array(vect.get_feature_names_out())

# Spaltennamen nach dem höchsten TF-IDF-Wert in der Spalte sortieren
sorted_tfidf_index = X_train_vectorized.toarray().max(0).argsort()

# Wörter mit den höchsten und niedrigsten TF-IDF-Werten ausgeben
print("Smallest tfidf:\n{}\n".format(feature_names[sorted_tfidf_index[:10]]))
print("Largest tfidf: \n{}".format(feature_names[sorted_tfidf_index[:-11:-1]]))

Nutzen wir unsere neuen Features für die logistische Regression.

In [ ]:
# Modell trainieren
model = LogisticRegression(max_iter=1500)
model.fit(X_train_vectorized, y_train)

# Vorhersage für die transformierten Testdokumente
predictions = model.predict_proba(vect.transform(X_test))[:, 1]

print("AUC = {:.3f}".format(roc_auc_score(y_test, predictions)))

**Ergebnis:** Mit TF-IDF und einer minimalen Dokumenthäufigkeit von 15 verwendet das Modell nur 577 Features (statt 7546), erreicht aber eine AUC von etwa 0,988, etwas höher als beim zählbasierten Modell. TF-IDF gewichtet Wörter, die in fast jeder Nachricht vorkommen, schwächer und Wörter, die für wenige Nachrichten kennzeichnend sind, stärker. Daher schneidet ein viel kleineres, aussagekräftigeres Feature-Set genauso gut oder besser ab.

Mit nur 577 von 7546 Features (unsere Trainingstexte enthielten 7546 verschiedene Wörter) erhalten wir immer noch einen hohen Wert für den AUC-Score.
Du kannst gern verschiedene Werte für die minimale Dokumenthäufigkeit des TF-IDF-Vectorizers testen und beobachten, wie sich das auf das Modell auswirkt.

Auch hier können wir uns die Koeffizienten unseres neuen Modells ansehen.


In [ ]:
# Koeffizienten des Modells sortieren
sorted_coef_index = model.coef_[0].argsort()

# Die 10 kleinsten und die 10 größten Koeffizienten finden
# Die 10 größten Koeffizienten werden mit [:-11:-1] indexiert,
# sodass die Liste vom größten zum kleinsten Wert sortiert ist
print("Smallest Coefs:\n{}\n".format(feature_names[sorted_coef_index[:10]]))
print("Largest Coefs: \n{}".format(feature_names[sorted_coef_index[:-11:-1]]))

Du siehst, dass sich die Features mit den höchsten und niedrigsten Koeffizienten im Vergleich zum vorherigen Modell stark überschneiden. Unabhängig davon, wie wir unseren Text in Features umwandeln, scheinen diese Wörter für die Klassifikation von Spam mit logistischer Regression wichtig zu sein.


### Du bist dran: Bigramme und eine höhere minimale Dokumenthäufigkeit ausprobieren

Bigramme (Paare benachbarter Wörter) können Wendungen wie „free entry“ oder „call now“ erfassen, die einzelnen Wörtern entgehen. Passe den TF-IDF-Vectorizer erneut an, sodass er Bigramme einbezieht, und beobachte, wie sich der Score verändert.

**Schritte**
1. Passe einen `TfidfVectorizer` mit `ngram_range=(1, 2)` und `min_df=5` an `X_train` an.
2. Wandle `X_train` um und trainiere eine `LogisticRegression(max_iter=1500)`.
3. Sage Wahrscheinlichkeiten für `X_test` vorher und gib die AUC aus.

**Tipp:** Die TF-IDF-Zelle oben ist dein Template; nur die Argumente des Vectorizers ändern sich.

**Erwartetes Ergebnis:** eine AUC im selben hohen Bereich wie bei den vorherigen Modellen (etwa 0,98 bis 0,99).

In [ ]:
# TODO 1: einen TfidfVectorizer mit Bigrammen (ngram_range=(1, 2)) und min_df=5 anpassen
vect_bigram = ...
X_train_bigram = ...

# TODO 2: eine logistische Regression mit den Bigramm-Features trainieren
model_bigram = ...

# TODO 3: Wahrscheinlichkeiten für den Testdatensatz vorhersagen, dann die AUC ausgeben
probs = ...
print(f"AUC = {roc_auc_score(y_test, probs):.3f}")

## Vorverarbeitung des Texts

Bisher war jede einzelne Wortform ein eigenes Feature. Wir können das Vokabular verkleinern und verwandte Wortformen zusammenfassen, indem wir die Wörter vor dem Zählen normalisieren. Wir probieren zwei klassische Ansätze aus: Stemming und Lemmatisierung. Beide führen verschiedene Oberflächenformen eines Worts auf ein gemeinsames Token zurück, wodurch die Features über die Nachrichten hinweg besser vergleichbar werden können.

### Stemming

Beim Stemming wird ein Wort auf seinen Wortstamm reduziert. Das Ergebnis ist für Menschen schlechter lesbar, macht den Text aber über die Beobachtungen hinweg besser vergleichbar.

Zum Beispiel haben die Wörter „consult“, „consultant“, „consulting“, „consultative“, „consultants“ denselben Wortstamm **„consult“**.

Wir fügen Stemming jetzt als Vorverarbeitungsschritt in unseren Workflow ein. Der PorterStemmer von nltk erzeugt die Wortstämme. Diese Features werden im CountVectorizer verwendet, um eine Matrix mit der Anzahl der Features (auf den Wortstamm reduzierte Wörter) zu erstellen.

In [ ]:
# Stemmer und CountVectorizer initialisieren
stemmer = nltk.PorterStemmer()
cv_analyzer = CountVectorizer().build_analyzer()
# tfidf_analyzer = TfidfVectorizer(min_df=15).build_analyzer()


def stemmed_words(doc):
    return (stemmer.stem(w) for w in cv_analyzer(doc))


# CountVectorizer mit der Stemming-Funktion definieren
stem_vectorizer = CountVectorizer(analyzer=stemmed_words)
# stem_vectorizer = TfidfVectorizer(min_df=15, analyzer = stemmed_words)


# X_train umwandeln
X_train_stem_vectorized = stem_vectorizer.fit_transform(X_train)

Zerlegen wir die Codezelle oben in Schritte, um zu zeigen, welcher Schritt was macht und warum wir das insgesamt tun:

- Die Funktion `build_analyzer()` von `CountVectorizer()` übernimmt die Vorverarbeitung, die Tokenisierung und die Erzeugung der N-Gramme für den Text
- In der Funktion `stemmed_words()`
  - Der Text wird zuerst durch `build_analyzer()` geleitet, dann wird jedes Wort im Text per Stemming auf seine Grundform reduziert
- Das Ganze wird im letzten Schritt ausgeführt, wenn wir `fit_transform()` auf dem Datensatz aufrufen

- In der Zelle unten sehen wir, wie `build_analyzer()` den Beispieltext vorverarbeitet und in Tokens zerlegt
- Und in der letzten Zeile reduziert der Stemmer jedes Wort im Text auf seine Grundform

In [ ]:
sample_text = X_train[:1]
print("Sample Text - ", sample_text[872])
print("-" * 30)
print("Text after passing through build_analyzer - ", cv_analyzer(sample_text[872]))
print("-" * 30)
print(
    "Text after stemming - ", [stemmer.stem(w) for w in cv_analyzer(sample_text[872])]
)

Du kannst auch ausprobieren, die Auskommentierung der TF-IDF-Zeilen in der Zelle oben aufzuheben, sodass du statt CountVectorizer TF-IDF verwendest

In [ ]:
# Modell mit dem auf Wortstämme reduzierten und vektorisierten Datensatz trainieren
model_stemm = LogisticRegression(max_iter=1500)
model_stemm.fit(X_train_stem_vectorized, y_train)

# Vorhersage für die transformierten Testdokumente
predictions = model_stemm.predict_proba(stem_vectorizer.transform(X_test))[:, 1]

print("AUC = {:.3f}".format(roc_auc_score(y_test, predictions)))

**Ergebnis:** Stemming reduziert das Vokabular auf etwa 6379 Features und ergibt eine AUC von etwa 0,984, ähnlich wie bei den vorherigen Modellen. Stemming ist ein grobes, regelbasiertes Abschneiden (es kann Nicht-Wörter wie `consult` aus `consultative` erzeugen), es opfert also etwas Lesbarkeit und Präzision für einen kleineren Feature-Raum. Hier wirkt es sich weder deutlich positiv noch deutlich negativ auf den Score aus.

In [ ]:
# Feature-Namen als NumPy-Array abrufen
feature_names = np.array(stem_vectorizer.get_feature_names_out())

# Koeffizienten des Modells sortieren
sorted_coef_index = model_stemm.coef_[0].argsort()

# Die 10 kleinsten und die 10 größten Koeffizienten finden
# Die 10 größten Koeffizienten werden mit [:-11:-1] indexiert,
# sodass die Liste vom größten zum kleinsten Wert sortiert ist
print("Smallest Coefs:\n{}\n".format(feature_names[sorted_coef_index[:10]]))
print("Largest Coefs: \n{}".format(feature_names[sorted_coef_index[:-11:-1]]))

Jetzt sehen wir unter den Features mit den betragsmäßig höchsten Koeffizienten einige andere Wörter. Egal, wie wir den Text in Features umwandeln: Eine kleine Gruppe starker Spamwörter dominiert das Modell weiterhin.

### Lemmatisierung

So wie wir Stemming verwendet haben, können wir auch Lemmatisierung auf unsere Daten anwenden.
Lemmatisierung reduziert Wortformen auf ihre Grundform (z. B. am, are, is --> be; car, cars, car's, cars' --> car).


In [ ]:
# Initialisierung
WNlemma = nltk.WordNetLemmatizer()
cv_analyzer = CountVectorizer().build_analyzer()
# cv_analyzer = TfidfVectorizer(min_df=15).build_analyzer()


def lemmatize_word(doc):
    return (WNlemma.lemmatize(t) for t in cv_analyzer(doc))


lemm_vectorizer = CountVectorizer(analyzer=lemmatize_word)
# lemm_vectorizer = TfidfVectorizer(min_df=15, analyzer=lemmatize_word)

# X_train umwandeln
X_train_lemm_vectorized = lemm_vectorizer.fit_transform(X_train)

In [ ]:
X_train_lemm_vectorized.shape

Mit der Lemmatisierung konnten wir die Features von ca. 7500 auf 7100 reduzieren.

In [ ]:
from sklearn.metrics import f1_score

In [ ]:
# Modell mit dem auf Wortstämme reduzierten und vektorisierten Datensatz trainieren
model_lemm = LogisticRegression(max_iter=1500)
model_lemm.fit(X_train_lemm_vectorized, y_train)

# Vorhersage für die transformierten Testdokumente
predict_probab = model_lemm.predict_proba(lemm_vectorizer.transform(X_test))[:, 1]
predictions = model_lemm.predict(lemm_vectorizer.transform(X_test))

print("F1 = {:.3f}".format(f1_score(y_test, predictions)))
print("AUC = {:.3f}".format(roc_auc_score(y_test, predict_probab)))

**Ergebnis:** Die Lemmatisierung behält etwa 7093 Features bei und ergibt eine AUC von etwa 0,984 mit einem F1-Score von etwa 0,909 für die Spamklasse. Die Lemmatisierung bildet Wörter auf echte Grundformen aus dem Wörterbuch ab (zum Beispiel `are` auf `be`), sodass die Tokens lesbar bleiben. Auf diesem Datensatz ist der Score ähnlich wie bei den anderen Repräsentationen: Die Wahl der Textvorverarbeitung spielt eine geringere Rolle als die Tatsache, dass einige wenige starke Spamwörter (etwa Wörter wie prize und claim) die Entscheidung dominieren.

In [ ]:
# Feature-Namen als NumPy-Array abrufen
feature_names = np.array(lemm_vectorizer.get_feature_names_out())


# Koeffizienten des Modells sortieren
sorted_coef_index = model_lemm.coef_[0].argsort()

# Die 10 kleinsten und die 10 größten Koeffizienten finden
# Die 10 größten Koeffizienten werden mit [:-11:-1] indexiert,
# sodass die Liste vom größten zum kleinsten Wert sortiert ist
print("Smallest Coefs:\n{}\n".format(feature_names[sorted_coef_index[:10]]))
print("Largest Coefs: \n{}".format(feature_names[sorted_coef_index[:-11:-1]]))

Die Ergebnisse dieses letzten Modells sind denen des ersten getesteten Modells sehr ähnlich.

Du kannst testen, wie gut Lemmatisierung in Kombination mit TF-IDF bei unseren Beispieldaten funktioniert. Entferne dazu einfach das `#` am Anfang der Zeile (vergiss nicht, `#` bei den entsprechenden Zeilen davor hinzuzufügen).

## Unseren Spamklassifikator testen

Testen wir unseren Klassifikator. Du kannst auch einen eigenen Text eingeben.

In [ ]:
# Deine Eingabe hier unten
input_text = "We went yesterday to the beach, call me first. But then also call the other guy pls. Use this number and then the other number"

# Oder ein Beispiel aus dem Testdatensatz verwenden
# input_text = X_test.sample(1).iloc[0]
input_text

In [ ]:
# Du kannst das Modell durch model_stemm oder model_lemm ersetzen
if model.predict(vect.transform([input_text]))[0] == 1:
    print("This is a spam!")
else:
    print("Not a spam :)")

In [ ]:
from sklearn.metrics import hamming_loss

print(f"The hamming loss is {hamming_loss(y_test, predictions):.4f}")

Du kannst gern eigene SMS testen und herausfinden, welche Wörter du hinzufügen kannst, damit die Vorhersage für eine Ham-Nachricht zu Spam wechselt.

## Zusammenfassung

In diesem Notebook hast du:

- mit `CountVectorizer` und `TfidfVectorizer` numerische Features aus dem Rohtext der SMS erstellt.
- einen Spamklassifikator mit logistischer Regression trainiert und mit dem AUC-Score evaluiert.
- Stemming und Lemmatisierung angewendet und ihre Wirkung auf die Features und den Score verglichen.
- das Modell anhand der Wörter mit den größten und kleinsten Koeffizienten interpretiert.

| Repräsentation | Features | AUC |
|---|---|---|
| Wortanzahl (Bag-of-Words) | 7546 | 0,985 |
| TF-IDF (min_df=15) | 577 | 0,988 |
| Wortanzahl + Stemming | 6379 | 0,984 |
| Wortanzahl + Lemmatisierung | 7093 | 0,984 |

Alle vier Repräsentationen erreichen hohe Scores. Das zeigt, dass auf diesem Datensatz bereits ein einfaches lineares Modell auf Basis von Wort-Features ein starker Spamfilter ist.

## Quellen & weiterführende Literatur

- [**Natural Language Processing with Python (the NLTK book)**](https://www.nltk.org/book/): Die Standardeinführung in die Textverarbeitung mit NLTK, einschließlich Stemming und Lemmatisierung.
- [**scikit-learn: Text feature extraction**](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction): Offizieller Leitfaden zu `CountVectorizer`, `TfidfVectorizer` und dem Bag-of-Words-Modell.